# Prepare MEDS Timelines for CLMBR

This notebook exports the required OMOP tables from the active All of Us CDR, converts them to MEDS, restricts them to the reconstructed analytic cohort, and removes every event occurring on or after each participant's lifestyle-survey index date.

The final output is `EHR_MEDS.parquet` in the configured data directory (default `~/lungcancer`). Generated OMOP and MEDS files contain controlled participant data and must remain inside the Workbench environment. Do not commit them to Git.

## 1. Configuration and prerequisites

In [ ]:
from pathlib import Path
import glob
import os
import shutil
import subprocess

import pandas as pd
import polars as pl
from google.cloud import bigquery

# Workbench 1 injected WORKSPACE_CDR and WORKSPACE_BUCKET. Workbench 2 exposes
# workspace-specific cloud resources through the `wb` CLI. Configure their IDs
# as environment variables after inspecting `wb resource list`.
CDR_RESOURCE_ID = os.environ.get("LUNG_CANCER_CDR_RESOURCE_ID", "").strip()
BUCKET_RESOURCE_ID = os.environ.get("LUNG_CANCER_BUCKET_RESOURCE_ID", "").strip()


def run_wb(*args):
    result = subprocess.run(
        ["wb", *args], check=True, capture_output=True, text=True
    )
    return result.stdout.strip()


def resolve_workbench_resource(resource_id):
    output = run_wb("resource", "resolve", f"--id={resource_id}")
    lines = [line.strip() for line in output.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(f"Workbench returned no location for resource {resource_id!r}.")
    return lines[-1].strip("`'\"")


CDR = os.environ.get("WORKSPACE_CDR", "").strip()
BUCKET = os.environ.get("WORKSPACE_BUCKET", "").strip()

if not CDR and CDR_RESOURCE_ID:
    CDR = resolve_workbench_resource(CDR_RESOURCE_ID)
if not BUCKET and BUCKET_RESOURCE_ID:
    BUCKET = resolve_workbench_resource(BUCKET_RESOURCE_ID)

if CDR:
    CDR = CDR.removeprefix("bq://").replace(":", ".", 1)
if BUCKET:
    BUCKET = BUCKET.rstrip("/")
    if not BUCKET.startswith("gs://"):
        BUCKET = "gs://" + BUCKET

if not CDR or not BUCKET:
    resource_listing = ""
    if shutil.which("wb"):
        try:
            resource_listing = run_wb("resource", "list")
        except Exception as exc:
            resource_listing = f"Could not run `wb resource list`: {exc}"
    raise RuntimeError(
        "Workbench 2 does not automatically set WORKSPACE_CDR/WORKSPACE_BUCKET.\n"
        "In a terminal run:  wb resource list\n"
        "Then set LUNG_CANCER_CDR_RESOURCE_ID to the CDR dataset resource ID and "
        "LUNG_CANCER_BUCKET_RESOURCE_ID to a controlled GCS bucket resource ID.\n\n"
        f"Current resources:\n{resource_listing}"
    )

if shutil.which("wb"):
    STORAGE_BACKEND = "wb"
elif shutil.which("gcloud"):
    STORAGE_BACKEND = "gcloud"
elif shutil.which("gsutil"):
    STORAGE_BACKEND = "gsutil"
else:
    raise RuntimeError("Need `wb`, `gcloud`, or `gsutil` to download exported Parquet files.")


def storage_copy(source, destination):
    if STORAGE_BACKEND == "wb":
        command = ["wb", "gcloud", "storage", "cp", source, str(destination)]
    elif STORAGE_BACKEND == "gcloud":
        command = ["gcloud", "storage", "cp", source, str(destination)]
    else:
        command = ["gsutil", "-m", "cp", source, str(destination)]
    subprocess.run(command, check=True)


DATA_DIR = Path(
    os.environ.get("LUNG_CANCER_DATA_DIR", Path.home() / "lungcancer")
).expanduser()
OMOP_DIR = DATA_DIR / "omop_parquet_lungcancer"
MEDS_RAW_DIR = DATA_DIR / "meds_output_lungcancer"
MEDS_FINAL = DATA_DIR / "EHR_MEDS.parquet"
COHORT_PATH = DATA_DIR / "paper_cohort.parquet"

TABLES_WITH_PERSON_ID = [
    "person", "drug_exposure", "visit_occurrence", "condition_occurrence",
    "death", "procedure_occurrence", "device_exposure", "measurement",
    "observation", "visit_detail", "note",
]
VOCAB_TABLES = ["concept", "concept_relationship"]
NUM_PROCESSES = min(16, os.cpu_count() or 4)

if not COHORT_PATH.exists():
    raise FileNotFoundError(f"Missing {COHORT_PATH}; run 01_Cohort_Construction.ipynb first.")
if shutil.which("meds_etl_omop") is None:
    raise RuntimeError('Install MEDS ETL first: pip install "meds_etl[cpp]==0.3.8"')

DATA_DIR.mkdir(parents=True, exist_ok=True)
OMOP_DIR.mkdir(parents=True, exist_ok=True)
print("CDR:", CDR)
print("Workspace bucket:", BUCKET)
print("Storage backend:", STORAGE_BACKEND)
print("OMOP directory:", OMOP_DIR)
print("Raw MEDS directory:", MEDS_RAW_DIR)
print("Final pre-index MEDS file:", MEDS_FINAL)


## 2. Define the source cohort filter

In [ ]:
# This is the original Lifestyle + EHR + ever-smoker source filter.
# Exact paper-cohort membership is applied locally after MEDS conversion.
W = f"`{CDR}`"
COHORT_SUBQUERY = f"""
SELECT DISTINCT person_id
FROM {W}.cb_search_person
WHERE has_ehr_data = 1
  AND person_id IN (
    SELECT DISTINCT person_id
    FROM {W}.cb_search_all_events
    WHERE concept_id IN (
      SELECT DISTINCT c.concept_id
      FROM {W}.cb_criteria c
      JOIN (
        SELECT CAST(cr.id AS STRING) AS id
        FROM {W}.cb_criteria cr
        WHERE concept_id IN (1585855)
          AND full_text LIKE '%_rank1]%'
      ) a
      ON c.path LIKE CONCAT('%.', a.id, '.%')
        OR c.path LIKE CONCAT('%.', a.id)
        OR c.path LIKE CONCAT(a.id, '.%')
        OR c.path = a.id
      WHERE is_standard = 0 AND is_selectable = 1
    )
    AND is_standard = 0
  )
  AND person_id IN (
    SELECT DISTINCT person_id
    FROM {W}.cb_search_all_events
    WHERE concept_id = 1585857
      AND is_standard = 0
      AND value_source_concept_id = 1585858
  )
"""

## 3. Export OMOP tables to the workspace bucket

In [ ]:
client = bigquery.Client()
EXPORT_PREFIX = f"{BUCKET}/lungcancer_omop"

def export_table(table, participant_filtered):
    uri = f"{EXPORT_PREFIX}/{table}/{table}-*.parquet"
    where = f"WHERE person_id IN ({COHORT_SUBQUERY})" if participant_filtered else ""
    sql = f"""
    EXPORT DATA OPTIONS (
      uri = '{uri}', format = 'PARQUET', overwrite = true
    ) AS
    SELECT * FROM `{CDR}.{table}`
    {where}
    """
    print("Exporting", table)
    client.query(sql).result()
    return uri

for table in TABLES_WITH_PERSON_ID:
    export_table(table, participant_filtered=True)
for table in VOCAB_TABLES:
    export_table(table, participant_filtered=False)

print("BigQuery exports complete")

## 4. Download into the directory structure expected by MEDS ETL

In [ ]:
for table in TABLES_WITH_PERSON_ID + VOCAB_TABLES:
    table_dir = OMOP_DIR / table
    table_dir.mkdir(parents=True, exist_ok=True)
    source = f"{EXPORT_PREFIX}/{table}/{table}-*.parquet"
    print("Downloading", table)
    storage_copy(source, table_dir)

for table in TABLES_WITH_PERSON_ID + VOCAB_TABLES:
    count = len(list((OMOP_DIR / table).glob("*.parquet")))
    if count == 0:
        raise FileNotFoundError(f"No downloaded Parquet shards for {table}")
    print(table, count, "files")


## 5. Convert OMOP to MEDS

In [ ]:
# The MEDS ETL release available in Workbench expects the newer OMOP visit
# names, while the AoU export uses the older names. Add aliases to the local
# exported copies; retain the source columns and values unchanged.
VISIT_COLUMN_ALIASES = {
    "admitting_source_concept_id": "admitted_from_concept_id",
    "admitting_source_value": "admitted_from_source_value",
    "discharge_to_concept_id": "discharged_to_concept_id",
    "discharge_to_source_value": "discharged_to_source_value",
}

for table in ["visit_occurrence", "visit_detail"]:
    for parquet_path in sorted((OMOP_DIR / table).glob("*.parquet")):
        schema = pl.read_parquet_schema(parquet_path)
        expressions = [
            pl.col(source).alias(destination)
            for source, destination in VISIT_COLUMN_ALIASES.items()
            if source in schema and destination not in schema
        ]
        if expressions:
            temporary_path = parquet_path.with_suffix(".compat.parquet")
            (
                pl.scan_parquet(parquet_path)
                .with_columns(expressions)
                .collect(streaming=True)
                .write_parquet(temporary_path, compression="zstd")
            )
            temporary_path.replace(parquet_path)
            print("Added MEDS-compatible aliases:", parquet_path.name)

if MEDS_RAW_DIR.exists() and any(MEDS_RAW_DIR.iterdir()):
    raise FileExistsError(
        f"{MEDS_RAW_DIR} already contains files. Reuse it by skipping this cell, "
        "or rename/remove that specific directory after verifying it is obsolete."
    )

subprocess.run([
    "meds_etl_omop",
    str(OMOP_DIR),
    str(MEDS_RAW_DIR),
    "--backend", "cpp",
    "--num_proc", str(NUM_PROCESSES),
    "--num_shards", str(NUM_PROCESSES),
], check=True)

raw_shards = sorted((MEDS_RAW_DIR / "data").glob("*.parquet"))
if not raw_shards:
    raise FileNotFoundError(f"No MEDS shards found under {MEDS_RAW_DIR / 'data'}")
print("Raw MEDS shards:", len(raw_shards))

## 6. Restrict to the paper cohort and truncate before index

In [ ]:
paper_cohort = pd.read_parquet(
    COHORT_PATH,
    columns=["person_id", "lifestyle_survey_date"],
).drop_duplicates("person_id")
paper_cohort["person_id"] = pd.to_numeric(paper_cohort["person_id"], errors="raise").astype("int64")
paper_cohort["lifestyle_survey_date"] = pd.to_datetime(
    paper_cohort["lifestyle_survey_date"], errors="raise", utc=True
).dt.tz_convert(None)

index_dates = (
    pl.from_pandas(paper_cohort)
    .rename({"person_id": "subject_id"})
    .with_columns(
        pl.col("subject_id").cast(pl.Int64),
        pl.col("lifestyle_survey_date").cast(pl.Datetime),
    )
)

raw = pl.scan_parquet([str(p) for p in raw_shards])
required = {"subject_id", "time", "code"}
missing = required - set(raw.collect_schema().names())
if missing:
    raise ValueError(f"MEDS output is missing required columns: {sorted(missing)}")

ppi_keep = (
    ~pl.col("code").str.starts_with("PPI/")
    | pl.col("code").str.starts_with("PPI/Gender")
    | pl.col("code").str.starts_with("PPI/WhatRace")
)

preindex = (
    raw
    .with_columns(
        pl.col("subject_id").cast(pl.Int64, strict=False),
        pl.col("time").cast(pl.Datetime, strict=False),
    )
    .join(index_dates.lazy(), on="subject_id", how="inner")
    .filter(pl.col("time").is_not_null())
    .filter(pl.col("time") < pl.col("lifestyle_survey_date"))
    .filter(ppi_keep)
    .drop("lifestyle_survey_date")
    .sort(["subject_id", "time"])
)

preindex.sink_parquet(MEDS_FINAL)
print("Saved:", MEDS_FINAL)

## 7. Leakage and coverage audit

In [ ]:
final = pl.scan_parquet(MEDS_FINAL)
audit = (
    final
    .join(index_dates.lazy(), on="subject_id", how="left")
    .select([
        pl.len().alias("events"),
        pl.col("subject_id").n_unique().alias("participants_with_events"),
        (pl.col("time") >= pl.col("lifestyle_survey_date")).sum().alias("events_on_or_after_index"),
        pl.col("time").min().alias("first_event"),
        pl.col("time").max().alias("last_event"),
    ])
    .collect()
)
display(audit.to_pandas())

leakage_events = int(audit["events_on_or_after_index"][0])
if leakage_events != 0:
    raise AssertionError(f"Temporal leakage detected: {leakage_events} events on/after index")

coverage = int(audit["participants_with_events"][0]) / len(paper_cohort)
print(f"Participant coverage: {coverage:.1%}")
print("Temporal leakage audit passed")

## Output contract

The next notebook consumes `EHR_MEDS.parquet` from the configured data directory. It may omit cohort members with fewer than five pre-index events; the CLMBR notebook records missing embeddings explicitly and imputes within each training fold.